In [ ]:
# load the liabries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# load the dataset
dataset = pd.read_feather('all_flights.feather')
display(dataset.head())
num_rows = len(dataset)
# iterate through all the columsn and if there are any nan values, we print the column name and the number of nan values
dataset_completed_flights = dataset[(dataset['Cancelled'] == 0) & (dataset['Diverted'] == 0)]
for column in dataset_completed_flights.columns:
    if dataset_completed_flights[column].isna().sum() > 0:
        # calcuate the percentage of nan values in the column
        percentage_nan = dataset_completed_flights[column].isna().sum() / num_rows * 100
        print(f"Column {column} has {dataset_completed_flights[column].isna().sum()} nan values ({percentage_nan:.2f}%)")
dataset_completed_flights = None

min_year = 2009


,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,TZ_Dest,CRSDepDateTime,CRSDepDateTime_UTC,CRSArrDateTime,CRSArrDateTime_UTC,DepDateTime,DepDateTime_UTC,ArrDateTime,ArrDateTime_UTC,DayOfYear
0,2009,1,13,2,2009-01-13,DL,N688DL,1237,ATL,RSW,...,America/New_York,2009-01-13 14:51:00,2009-01-13 19:51:00+00:00,2009-01-13 16:36:00,2009-01-13 21:36:00+00:00,2009-01-13 14:47:00,2009-01-13 19:47:00+00:00,2009-01-13 16:56:00,2009-01-13 21:56:00+00:00,13
1,2009,1,13,2,2009-01-13,DL,N3752,1239,SLC,PDX,...,America/Los_Angeles,2009-01-13 20:20:00,2009-01-14 03:20:00+00:00,2009-01-13 21:27:00,2009-01-14 05:27:00+00:00,2009-01-13 20:27:00,2009-01-14 03:27:00+00:00,2009-01-13 21:27:00,2009-01-14 05:27:00+00:00,13
2,2009,1,13,2,2009-01-13,DL,N913DN,1240,SLC,ORD,...,America/Chicago,2009-01-13 10:55:00,2009-01-13 17:55:00+00:00,2009-01-13 15:23:00,2009-01-13 21:23:00+00:00,2009-01-13 10:52:00,2009-01-13 17:52:00+00:00,2009-01-13 15:07:00,2009-01-13 21:07:00+00:00,13
3,2009,1,13,2,2009-01-13,DL,N915DN,1241,DTW,SLC,...,America/Denver,2009-01-13 16:40:00,2009-01-13 21:40:00+00:00,2009-01-13 18:43:00,2009-01-14 01:43:00+00:00,2009-01-13 16:38:00,2009-01-13 21:38:00+00:00,2009-01-13 18:33:00,2009-01-14 01:33:00+00:00,13
4,2009,1,13,2,2009-01-13,DL,N915DN,1242,BOI,SLC,...,America/Denver,2009-01-13 07:55:00,2009-01-13 14:55:00+00:00,2009-01-13 09:07:00,2009-01-13 16:07:00+00:00,2009-01-13 07:49:00,2009-01-13 14:49:00+00:00,2009-01-13 09:03:00,2009-01-13 16:03:00+00:00,13


Column CarrierDelay has 5458466 nan values (80.01%)
Column WeatherDelay has 5458466 nan values (80.01%)
Column NASDelay has 5458466 nan values (80.01%)
Column SecurityDelay has 5458466 nan values (80.01%)
Column LateAircraftDelay has 5458466 nan values (80.01%)


In [5]:
# features to calcualte the averge delay fro each departure airport, in every month
df_delays = dataset.groupby(['Origin', dataset['Month'], dataset['Year']])['ArrDelay'].mean().reset_index()
# change the name of the columns
df_delays = df_delays.rename(columns={'ArrDelay': 'prev_AvgArrDelay'})
# fill the nan values with 0, as if there are no flights in the previous month, we can assume that the average delay is 0
# change the key, to the next month, so our feature will be the average delay of the previous month, as we can not use the average delay of the current month, as it would be a data leak
df_delays['Month'] = df_delays['Month'] + 1
mask = (df_delays['Month'] > 12).astype(int)
df_delays['Year'] = df_delays['Year'] + mask
df_delays['Month'] = df_delays['Month'] - mask * 12


# we merge the average delay with the original dataset, on the Origin, month and year of the departure time
dataset = dataset.merge(df_delays, left_on=['Origin', dataset['CRSDepDateTime'].dt.month, dataset['CRSDepDateTime'].dt.year], right_on=['Origin', 'Month', 'Year'], how='left',suffixes=('', '_AvgDelay'))
# we drop the month and year columns
dataset = dataset.drop(columns=['Month_AvgDelay', 'Year_AvgDelay'])
display(dataset)

,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,CRSDepDateTime_UTC,CRSArrDateTime,CRSArrDateTime_UTC,DepDateTime,DepDateTime_UTC,ArrDateTime,ArrDateTime_UTC,DayOfYear,knownWeatherDateTime_UTC,prev_AvgArrDelay
0,2009,1,13,2,2009-01-13,DL,N688DL,1237,ATL,RSW,...,2009-01-13 19:51:00+00:00,2009-01-13 16:36:00,2009-01-13 21:36:00+00:00,2009-01-13 14:47:00,2009-01-13 19:47:00+00:00,2009-01-13 16:56:00,2009-01-13 21:56:00+00:00,13,2009-01-13 17:00:00,15.289502
1,2009,1,13,2,2009-01-13,DL,N3752,1239,SLC,PDX,...,2009-01-14 03:20:00+00:00,2009-01-13 21:27:00,2009-01-14 05:27:00+00:00,2009-01-13 20:27:00,2009-01-14 03:27:00+00:00,2009-01-13 21:27:00,2009-01-14 05:27:00+00:00,13,2009-01-14 01:00:00,17.375842
2,2009,1,13,2,2009-01-13,DL,N913DN,1240,SLC,ORD,...,2009-01-13 17:55:00+00:00,2009-01-13 15:23:00,2009-01-13 21:23:00+00:00,2009-01-13 10:52:00,2009-01-13 17:52:00+00:00,2009-01-13 15:07:00,2009-01-13 21:07:00+00:00,13,2009-01-13 15:00:00,17.375842
3,2009,1,13,2,2009-01-13,DL,N915DN,1241,DTW,SLC,...,2009-01-13 21:40:00+00:00,2009-01-13 18:43:00,2009-01-14 01:43:00+00:00,2009-01-13 16:38:00,2009-01-13 21:38:00+00:00,2009-01-13 18:33:00,2009-01-14 01:33:00+00:00,13,2009-01-13 19:00:00,25.660777
4,2009,1,13,2,2009-01-13,DL,N915DN,1242,BOI,SLC,...,2009-01-13 14:55:00+00:00,2009-01-13 09:07:00,2009-01-13 16:07:00+00:00,2009-01-13 07:49:00,2009-01-13 14:49:00+00:00,2009-01-13 09:03:00,2009-01-13 16:03:00+00:00,13,2009-01-13 12:00:00,16.062678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6822381,2008,12,31,3,2008-12-31,WN,N730SW,2037,TUS,MDW,...,2008-12-31 13:30:00+00:00,2008-12-31 10:45:00,2008-12-31 16:45:00+00:00,2008-12-31 06:45:00,2008-12-31 13:45:00+00:00,2008-12-31 10:52:00,2008-12-31 16:52:00+00:00,366,2008-12-31 11:00:00,1.224469
6822382,2008,12,31,3,2008-12-31,WN,N278WN,1005,TUS,SAN,...,2009-01-01 02:25:00+00:00,2008-12-31 19:40:00,2009-01-01 03:40:00+00:00,2008-12-31 19:23:00,2009-01-01 02:23:00+00:00,2008-12-31 19:31:00,2009-01-01 03:31:00+00:00,366,2009-01-01 00:00:00,1.224469
6822383,2008,12,31,3,2008-12-31,WN,N391SW,2540,TUS,SAN,...,2008-12-31 18:40:00+00:00,2008-12-31 11:55:00,2008-12-31 19:55:00+00:00,2008-12-31 11:40:00,2008-12-31 18:40:00+00:00,2008-12-31 11:45:00,2008-12-31 19:45:00+00:00,366,2008-12-31 16:00:00,1.224469
6822384,2008,12,31,3,2008-12-31,WN,N652SW,2557,TUS,SAN,...,2008-12-31 14:35:00+00:00,2008-12-31 07:50:00,2008-12-31 15:50:00+00:00,2008-12-31 07:34:00,2008-12-31 14:34:00+00:00,2008-12-31 07:40:00,2008-12-31 15:40:00+00:00,366,2008-12-31 12:00:00,1.224469


In [6]:
airport_limit_list =["JFK","LAX","MIA","SFO","EWR","ORD","ATL","DFW","IAH",
"BOS","MCO","FLL","SEA","CLT","DEN","PHL","LAS","HNL","DTW","MSP","PHX","LGA","TPA",
"SLC","BWI","AUS","SAN","HOU","PDX","MDW","OAK","BNA","DCA","STL","DAL"]

In [7]:
dataset_airport_limit = dataset[
    dataset['Origin'].isin(airport_limit_list)
    & dataset['Dest'].isin(airport_limit_list)
    & (dataset['Cancelled'] == 0)
    & (dataset['Year'] >= min_year)
].copy()

# Source rows used to compute historical delay at each airport/day
src = dataset[['Origin', 'FlightDate', 'DepDateTime', 'DepDelayMinutes','Cancelled','Year','FlightId']].copy()
# drop all rows where the flight was in a year to early for our dataset, or where the flight was cancelled, as we can not use these flights to calculate the historical delay at each airport/day
src = src[(src['Year'] >= min_year) & (src['Origin'].isin(airport_limit_list))]
#


src['DepDelayMinutes'] = src['DepDelayMinutes'].fillna(0)
src['count'] = -(src['Cancelled'] - 1)
# Cumulative delay stats per airport-day
src = src.sort_values(['Origin', 'FlightDate', 'DepDateTime'])
grp = src.groupby(['Origin', 'FlightDate'], sort=False)
src['cum_cancelled'] = grp['Cancelled'].cumsum()
src['cum_delay'] = grp['DepDelayMinutes'].cumsum()
src['cum_count'] = grp['count'].cumsum()
# calculate the average delay for each airport/day, by dividing the cumulative delay by the cumulative count of flights
# if there are no flights, we set the average delay to 0, to avoid division by zero
src['avg_delay'] = src['cum_delay'] / src['cum_count'] 
src['avg_delay'] = src['avg_delay'].fillna(0)

# drop cancelled flights, as we can not use these flights to calculate the historical delay at each airport/day
src = src[src['Cancelled'] == 0]

# group the information by Origin, Flight date, and the next hour (ceil hour) of the departure time, to get the cumulative delay and count of flights for each airport, here always use the row where the time is the latest or the count is highest.
src['DepDateTime_ceil'] = src['DepDateTime'].dt.ceil('h')
src = src.sort_values(['Origin', 'FlightDate', 'DepDateTime_ceil', 'DepDateTime'], ascending=[True, True, True, False])
src = src.drop_duplicates(subset=['Origin', 'FlightDate', 'DepDateTime_ceil'], keep='first')
grp = src.groupby(['Origin', 'FlightDate', 'DepDateTime_ceil'], sort=False)
# for every hour i want the last information about the cumulative delay and count of flights, so i want to get the last row for each group
src = grp.last().reset_index()
# also get the number of flights that departed in the last hour, to get the cumulative count of flights for each airport, here always use the row where the time is the latest or the count is highest.
src['flights_in_last_hour'] = grp['count'].sum().reset_index(drop=True)


dataset_airport_limit['floor_informationtime'] = dataset_airport_limit['DepDateTime'].dt.floor('h') - pd.to_timedelta(hours_before, unit='h')
# the columsn we still need of src: 
src = src[['Origin', 'FlightDate', 'DepDateTime_ceil', 'avg_delay', 'cum_count','cum_cancelled', 'flights_in_last_hour']].copy()
# we merge the information of src with the original dataset, by merging on the Origin, FlightDate and the next hour (ceil hour) of the departure time, to get the cumulative delay and count of flights for each airport, here always use the row where the time is the latest or the count is highest.
dataset_airport_limit = dataset_airport_limit.merge(src, left_on=['Origin', 'FlightDate', 'floor_informationtime'], right_on=['Origin', 'FlightDate', 'DepDateTime_ceil'], how='left',suffixes=('', '_src'))
# fill the nan values with 0, as if there are no flights in the previous hour, we can assume that the average delay is 0 and the count of flights is 0
dataset_airport_limit['avg_delay'] = dataset_airport_limit['avg_delay'].fillna(0)
dataset_airport_limit['cum_count'] = dataset_airport_limit['cum_count'].fillna(0)
dataset_airport_limit['cum_cancelled'] = dataset_airport_limit['cum_cancelled'].fillna(0)
dataset_airport_limit['flights_in_last_hour'] = dataset_airport_limit['flights_in_last_hour'].fillna(0)
# we drop the columns we do not need anymore
dataset_airport_limit2 = dataset_airport_limit.drop(columns=['DepDateTime_ceil','floor_informationtime'])

dataset_airport_limit2 = dataset_airport_limit2[["FlightId", "avg_delay", "cum_count","cum_cancelled", "flights_in_last_hour"]]
# we merge the information of src with the original dataset, by merging on the Origin, FlightDate and the next hour (ceil hour) of the departure time, to get the cumulative delay and count of flights for each airport, here always use the row where the time is the latest or the count is highest.
dataset = dataset.merge(dataset_airport_limit2, left_on='FlightId', right_on='FlightId', how='left')
display(dataset_airport_limit)


,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,ArrDateTime_UTC,DayOfYear,knownWeatherDateTime_UTC,prev_AvgArrDelay,floor_informationtime,DepDateTime_ceil,avg_delay,cum_count,cum_cancelled,flights_in_last_hour
0,2009,1,13,2,2009-01-13,DL,N3752,1239,SLC,PDX,...,2009-01-14 05:27:00+00:00,13,2009-01-14 01:00:00,17.375842,2009-01-13 18:00:00,2009-01-13 18:00:00,4.828829,111.0,0.0,1.0
1,2009,1,13,2,2009-01-13,DL,N913DN,1240,SLC,ORD,...,2009-01-13 21:07:00+00:00,13,2009-01-13 15:00:00,17.375842,2009-01-13 08:00:00,2009-01-13 08:00:00,0.230769,13.0,0.0,1.0
2,2009,1,13,2,2009-01-13,DL,N915DN,1241,DTW,SLC,...,2009-01-14 01:33:00+00:00,13,2009-01-13 19:00:00,25.660777,2009-01-13 14:00:00,NaT,0.000000,0.0,0.0,0.0
3,2009,1,13,2,2009-01-13,DL,N915DN,1242,SLC,DTW,...,2009-01-13 20:36:00+00:00,13,2009-01-13 15:00:00,17.375842,2009-01-13 07:00:00,2009-01-13 07:00:00,0.333333,9.0,0.0,1.0
4,2009,1,13,2,2009-01-13,DL,N663DN,1260,SEA,SLC,...,2009-01-13 15:40:00+00:00,13,2009-01-13 12:00:00,22.693852,2009-01-13 03:00:00,NaT,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3102351,2010,12,28,2,2010-12-28,WN,N693SW,1424,OAK,SEA,...,2010-12-28 19:01:00+00:00,362,2010-12-28 15:00:00,6.220498,2010-12-28 07:00:00,2010-12-28 07:00:00,8.000000,13.0,0.0,1.0
3102352,2010,12,28,2,2010-12-28,WN,N315SW,423,OAK,SLC,...,2010-12-29 01:10:00+00:00,362,2010-12-28 20:00:00,6.220498,2010-12-28 13:00:00,2010-12-28 13:00:00,15.813559,59.0,0.0,1.0
3102353,2010,12,28,2,2010-12-28,WN,N287WN,1152,OAK,SLC,...,2010-12-29 06:53:00+00:00,362,2010-12-29 02:00:00,6.220498,2010-12-28 19:00:00,2010-12-28 19:00:00,21.881720,93.0,0.0,1.0
3102354,2010,12,28,2,2010-12-28,WN,N703SW,2445,OAK,SLC,...,2010-12-28 20:35:00+00:00,362,2010-12-28 17:00:00,6.220498,2010-12-28 09:00:00,2010-12-28 09:00:00,8.346154,26.0,0.0,1.0


In [8]:
# build an Id for each flight, by combining the Reporting_Airline, the Flight_Number_Reporting_Airline and the FlightDate and the Hour of the departure time
# check if this is truly a unique identifier
print("Number of unique FlightIds: ", dataset['FlightId'].nunique())
print("Number of rows in the dataset: ", len(dataset))

# it seems to not be unique, to investigate, we sort by the FlightId and check for duplicates

# we sort the dataset by tail number and flight date, to make it easier to find the previous flight
dataset = dataset.sort_values(by=['Tail_Number', 'CRSDepDateTime_UTC'])
display(dataset)

Number of unique FlightIds:  6822386
Number of rows in the dataset:  6822386


,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,DepDateTime_UTC,ArrDateTime,ArrDateTime_UTC,DayOfYear,knownWeatherDateTime_UTC,prev_AvgArrDelay,avg_delay,cum_count,cum_cancelled,flights_in_last_hour
5509551,2010,10,1,5,2010-10-01,B6,N-504J,37,BUF,JFK,...,2010-10-01 12:01:00+00:00,2010-10-01 09:30:00,2010-10-01 13:30:00+00:00,274,2010-10-01 07:00:00,0.592328,NaN,NaN,NaN,NaN
1613129,2009,7,1,3,2009-07-01,B6,N-J350,647,JFK,SFO,...,2009-07-01 23:23:00+00:00,2009-07-01 23:46:00,2009-07-02 06:46:00+00:00,182,2009-07-01 19:00:00,11.166992,13.528169,142.0,1.0,1.0
1617523,2009,7,9,4,2009-07-09,B6,N-J350,148,JFK,BTV,...,2009-07-10 01:11:00+00:00,2009-07-09 22:08:00,2009-07-10 02:08:00+00:00,190,2009-07-09 22:00:00,11.166992,NaN,NaN,NaN,NaN
1625658,2009,7,23,4,2009-07-23,B6,N-J350,426,PBI,BOS,...,2009-07-23 20:52:00+00:00,2009-07-23 19:49:00,2009-07-23 23:49:00+00:00,204,2009-07-23 18:00:00,17.781553,NaN,NaN,NaN,NaN
1629905,2009,7,31,5,2009-07-31,B6,N-J350,307,IAD,LGB,...,2009-08-01 01:31:00+00:00,2009-07-31 23:45:00,2009-08-01 06:45:00+00:00,212,2009-07-31 20:00:00,14.888270,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5965615,2010,12,30,4,2010-12-30,UA,NaN,258,DEN,ORD,...,NaT,NaT,NaT,364,2010-12-31 02:00:00,-0.240415,NaN,NaN,NaN,NaN
5958014,2010,12,30,4,2010-12-30,UA,NaN,9,DEN,SAN,...,NaT,NaT,NaT,364,2010-12-31 02:00:00,-0.240415,NaN,NaN,NaN,NaN
5957345,2010,12,30,4,2010-12-30,F9,NaN,788,DEN,MKE,...,NaT,NaT,NaT,364,2010-12-31 02:00:00,-0.240415,NaN,NaN,NaN,NaN
5972001,2010,12,31,5,2010-12-31,UA,NaN,468,SEA,SFO,...,NaT,NaT,NaT,365,2010-12-31 12:00:00,0.913995,NaN,NaN,NaN,NaN


In [9]:
# report if there are any nans int the df
print("Number of NaN values in each column:")
print(dataset.isna().sum())

Number of NaN values in each column:
Year                                     0
Month                                    0
DayofMonth                               0
DayOfWeek                                0
FlightDate                               0
Reporting_Airline                        0
Tail_Number                          17113
Flight_Number_Reporting_Airline          0
Origin                                   0
Dest                                     0
CRSDepTime                               0
DepTime                              86531
DepDelay                             86531
DepDelayMinutes                      86531
TaxiOut                              88431
WheelsOff                            88431
WheelsOn                             89028
TaxiIn                               91934
CRSArrTime                           16133
ArrTime                             105161
ArrDelay                            105161
ArrDelayMinutes                     105161
Cancelled        

In [10]:
# # get for every flight the avergage delay of every flight staring from the same airport on that day, unitl 2 hours before sheduled departure time.

# # copy dataset, with only flight where the origin is in the airport limit list
# dataset_airport_limit = dataset[dataset['Origin'].isin(airport_limit_list)& dataset['Dest'].isin(airport_limit_list)].copy()

# # for evyery flight here we want to get the average delay of all flights starting from the same airport on the same day, until 2 hours before sheduled departure time.
# for index, row in dataset_airport_limit.iterrows():
#     # get the departure airport and the departure time of the flight
#     departure_airport = row['Origin']
#     local_2_hours_before_departure = row['CRSDepDateTime'] - pd.to_timedelta(2, unit='h')
#     # get all flights starting from the same airport on the same day, until 2 hours before sheduled departure time. 
#     mask = airport_masks[departure_airport] & (dataset['CRSDepDateTime'] <= local_2_hours_before_departure) & (dataset['DayOfYear'] == row['DayOfYear']) & (dataset['Year'] == row['Year'])
#     # if there where no flights that day so far, the means the average delay is 0, otherwise we calculate the average delay of these flights
#     avg_delay = dataset.loc[mask, 'DepDelay'].mean()
#     # assign the average delay to the flight
#     dataset.at[index, 'AvgDelaySameDaySameAirport'] = avg_delay

# # delete the airport masks to save memory
# del airport_masks

In [11]:
# finding the previous flight for each aircraft. 
# naivly for now take the FlightID of the row above.
dataset['tmp_PreviousFlightId'] = dataset['FlightId'].shift(1)
# get the arrival airport of the previous flight, from the row above
dataset['PreviousFlightDest'] = dataset['Dest'].shift(1)
#  also get airline and Tail number of the previous flight
dataset['PreviousFlightAirline'] = dataset['Reporting_Airline'].shift(1)
dataset['PreviousFlightTailNumber'] = dataset['Tail_Number'].shift(1)
# also get if the previous flight was cancelled or diverted, to exclude these flights later
dataset['PreviousFlightCancelled'] = dataset['Cancelled'].shift(1)
dataset['PreviousFlightDiverted'] = dataset['Diverted'].shift(1)

# build a mask to see if the airlines match, if the tail number matches and if the departure airport of the current flight matches the arrival airport of the previous flight
mask = (dataset['Tail_Number'] == dataset['PreviousFlightTailNumber']) & (dataset['Reporting_Airline'] == dataset['PreviousFlightAirline']) & (dataset['Origin'] == dataset['PreviousFlightDest'])
# also exclude the previous flight if it was cancelled or diverted
mask = mask & (dataset['PreviousFlightCancelled'] == 0) & (dataset['PreviousFlightDiverted'] == 0)

# apply the mask 
dataset['PreviousFlightId'] = dataset['tmp_PreviousFlightId'].where(mask, other=np.nan)

# drop the temporary columns 
dataset = dataset.drop(columns=['tmp_PreviousFlightId', 'PreviousFlightDest', 'PreviousFlightAirline', 'PreviousFlightTailNumber', 'PreviousFlightCancelled', 'PreviousFlightDiverted'])


# now that i have the previous flights information we drop the first month of our dataset, as we only wanted it for the previous flight information.
dataset = dataset[dataset['Year'] >= min_year]
# introduce if this is the first flight record for the aircraft, by checking if the PreviousFlightId is null
dataset['FirstFlightRecord'] = dataset['PreviousFlightId'].isna().astype(int)


In [12]:
# create a subeset of columns relevant for joingg with the original dataset to get the departure and arrival times of the previous flight
previous_flights = dataset[['FlightId','CRSArrDateTime_UTC','ArrDateTime_UTC']]
print("1")
# rename the columns to indicate that they are the departure and arrival times of the previous flight
previous_flights = previous_flights.rename(columns={'CRSArrDateTime_UTC': 'PreviousFlightArrDateTime_UTC', 'ArrDateTime_UTC': 'PreviousFlightArrDateTime'})
print('2')
# join the previous_flights dataset with the original dataset, to get the departure and arrival times of the previous flight
dataset = dataset.merge(previous_flights, left_on='PreviousFlightId', right_on='FlightId', how='left', suffixes=('', '_PreviousFlight'))
print("3")
# calculate the turnaround time in minutes, by taking the difference between the departure time of the current flight and the arrival time of the previous flight
dataset['TurnaroundTime'] = (dataset['CRSDepDateTime_UTC'] - dataset['PreviousFlightArrDateTime_UTC']).dt.total_seconds() / 60
# fro analysis, drop later
# dataset['actual_TurnaroundTime'] = (dataset['DepDateTime_UTC'] - dataset['PreviousFlightArrDateTime_UTC']).dt.total_seconds() / 60

display(dataset[['FlightId', 'PreviousFlightId', 'CRSDepDateTime_UTC', 'PreviousFlightArrDateTime_UTC', 'TurnaroundTime']].head(20))


1
2
3


,FlightId,PreviousFlightId,CRSDepDateTime_UTC,PreviousFlightArrDateTime_UTC,TurnaroundTime
0,5509551,NaN,2010-10-01 09:50:00+00:00,NaT,NaN
1,1613129,NaN,2009-07-01 21:56:00+00:00,NaT,NaN
2,1617523,NaN,2009-07-10 00:12:00+00:00,NaT,NaN
3,1625658,NaN,2009-07-23 20:24:00+00:00,NaT,NaN
4,1629905,NaN,2009-07-31 22:16:00+00:00,NaT,NaN
5,1622621,NaN,2009-07-18 19:15:00+00:00,NaT,NaN
6,1625435,NaN,2009-07-23 21:00:00+00:00,NaT,NaN
7,3456331,NaN,2010-03-02 11:00:00+00:00,NaT,NaN
8,1616626,NaN,2009-07-07 15:20:00+00:00,NaT,NaN
9,3459276,NaN,2010-03-07 19:25:00+00:00,NaT,NaN


In [13]:
# how manny nan values are in the TurnaroundTime column
num_nan = dataset['TurnaroundTime'].isna().sum()
print("Number of NaN values in the TurnaroundTime column: ", num_nan)

# how manny times the tailnumber is missing in the dataset
num_nan_tailnumber = dataset['Tail_Number'].isna().sum()
print("Number of NaN values in the Tail_Number column: ", num_nan_tailnumber)
# mean tournaround time
mean_turnaround_time = dataset['TurnaroundTime'].mean()
print("Mean turnaround time: ", mean_turnaround_time)

# fill the nan values in the TurnaroundTime column with the mean turnaround time
dataset['TurnaroundTime'] = dataset['TurnaroundTime'].fillna(mean_turnaround_time)

# display(dataset)
# show all rows where the turnaround time is less than 0 and the previous row
# set all values for negative DepDelay to 0

planned_neg_tournaround = dataset['TurnaroundTime']  < -15
# drop all instances where the actual turnaround are more than 0
# actual_neg_tournaround = dataset['actual_TurnaroundTime'] < -10



# merge the two mask with an and
neg_tournaround = planned_neg_tournaround 

# make a mask for all cancellled flights
cancellled_flights = dataset['Cancelled'] == 1
diverted_flights = dataset['Diverted'] == 1

# remove all cancelled or diverted flights from the neg_tournaround mask
neg_tournaround = neg_tournaround & ~cancellled_flights & ~diverted_flights

# print the number of flights with an actual turnaround time of less than -10 minutes and a planned turnaround time of less than -15 minutes, that are not cancelled or diverted
actual_neg_tournaround2 =  ~cancellled_flights & ~diverted_flights
print("num_actual_neg_tournaround: ", actual_neg_tournaround2.sum())

# apply the combined mask to the dataset
neg_tournaround_dataset = dataset[neg_tournaround]
display(neg_tournaround_dataset)

# filter the dataset for values of turnaround time that are less than 0, as these are likely to be errors in the data
# neg_tournaround_dataset = dataset[dataset['TurnaroundTime'] <= 0]
# display(neg_tournaround_dataset)
# drop the actual_TurnaroundTime column, as it is not needed anymore
# dataset = dataset.drop(columns=['actual_TurnaroundTime'])


Number of NaN values in the TurnaroundTime column:  172428
Number of NaN values in the Tail_Number column:  14562
Mean turnaround time:  234.48522300476648
num_actual_neg_tournaround:  5976006


,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,avg_delay,cum_count,cum_cancelled,flights_in_last_hour,PreviousFlightId,FirstFlightRecord,FlightId_PreviousFlight,PreviousFlightArrDateTime_UTC,PreviousFlightArrDateTime,TurnaroundTime
39,2009,7,26,7,2009-07-26,B6,N-J363,160,MCO,JFK,...,12.668317,202.0,0.0,1.0,1627137.0,0,1627137.0,2009-07-27 00:37:00+00:00,2009-07-27 04:12:00+00:00,-124.0
244,2009,1,2,5,2009-01-02,AA,N058AA,1555,JFK,MIA,...,5.718182,110.0,0.0,1.0,184834.0,0,184834.0,2009-01-02 21:00:00+00:00,2009-01-02 20:45:00+00:00,-65.0
395,2009,1,17,6,2009-01-17,AA,N059AA,588,SJU,MIA,...,NaN,NaN,NaN,NaN,189892.0,0,189892.0,2009-01-17 21:35:00+00:00,2009-01-17 21:23:00+00:00,-145.0
984,2009,3,13,5,2009-03-13,AA,N065AA,1639,JFK,SJU,...,NaN,NaN,NaN,NaN,512813.0,0,512813.0,2009-03-13 21:40:00+00:00,2009-03-13 22:15:00+00:00,-30.0
1622,2009,1,17,6,2009-01-17,AA,N073AA,1692,SJU,JFK,...,NaN,NaN,NaN,NaN,190019.0,0,190019.0,2009-01-18 02:20:00+00:00,2009-01-18 02:04:00+00:00,-205.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6053560,2009,12,14,1,2009-12-14,DL,N999DN,54,IAH,ATL,...,0.000000,0.0,0.0,0.0,2881885.0,0,2881885.0,2009-12-15 02:20:00+00:00,2009-12-15 03:52:00+00:00,-120.0
6053647,2010,1,14,4,2010-01-14,DL,N999DN,2025,EWR,ATL,...,1.177778,45.0,0.0,1.0,2996443.0,0,2996443.0,2010-01-15 00:59:00+00:00,2010-01-15 00:54:00+00:00,-39.0
6053865,2010,3,15,1,2010-03-15,DL,N999DN,1978,ATL,DCA,...,5.583548,389.0,0.0,1.0,3492115.0,0,3492115.0,2010-03-16 00:50:00+00:00,2010-03-16 01:04:00+00:00,-30.0
6054340,2010,8,18,3,2010-08-18,DL,N999DN,2779,LGA,FLL,...,3.267717,127.0,1.0,1.0,4986538.0,0,4986538.0,2010-08-18 23:30:00+00:00,2010-08-18 23:20:00+00:00,-35.0


In [14]:


correlation = dataset['TurnaroundTime'].corr(dataset['DepDelay'])
print("Correlation between turnaround time and departure delay: ", correlation)

# smae thing for the arrival delay
correlation = dataset['TurnaroundTime'].corr(dataset['ArrDelay'])
print("Correlation between turnaround time and arrival delay: ", correlation)



Correlation between turnaround time and departure delay:  -0.009621571368561068
Correlation between turnaround time and arrival delay:  -0.007395891894626742


In [15]:
# list all columns in the dataset
print(dataset.columns)
#  remove the colums that are a refrence to the previous flight, as they are not needed anymore
dataset = dataset.drop(columns=['PreviousFlightId', 'PreviousFlightArrDateTime_UTC', 'PreviousFlightArrDateTime','FlightId_PreviousFlight'])

display(dataset)


Index(['Year', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
       'Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline',
       'Origin', 'Dest', 'CRSDepTime', 'DepTime', 'DepDelay',
       'DepDelayMinutes', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn',
       'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'Cancelled',
       'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime',
       'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay',
       'SecurityDelay', 'LateAircraftDelay', 'FlightId', 'Timezone_Origin',
       'TZ_Origin', 'Timezone_Dest', 'TZ_Dest', 'CRSDepDateTime',
       'CRSDepDateTime_UTC', 'CRSArrDateTime', 'CRSArrDateTime_UTC',
       'DepDateTime', 'DepDateTime_UTC', 'ArrDateTime', 'ArrDateTime_UTC',
       'DayOfYear', 'knownWeatherDateTime_UTC', 'prev_AvgArrDelay',
       'avg_delay', 'cum_count', 'cum_cancelled', 'flights_in_last_hour',
       'PreviousFlightId', 'FirstFlightRecord', 'FlightId_PreviousFlig

,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,ArrDateTime_UTC,DayOfYear,knownWeatherDateTime_UTC,prev_AvgArrDelay,avg_delay,cum_count,cum_cancelled,flights_in_last_hour,FirstFlightRecord,TurnaroundTime
0,2010,10,1,5,2010-10-01,B6,N-504J,37,BUF,JFK,...,2010-10-01 13:30:00+00:00,274,2010-10-01 07:00:00,0.592328,NaN,NaN,NaN,NaN,1,234.485223
1,2009,7,1,3,2009-07-01,B6,N-J350,647,JFK,SFO,...,2009-07-02 06:46:00+00:00,182,2009-07-01 19:00:00,11.166992,13.528169,142.0,1.0,1.0,1,234.485223
2,2009,7,9,4,2009-07-09,B6,N-J350,148,JFK,BTV,...,2009-07-10 02:08:00+00:00,190,2009-07-09 22:00:00,11.166992,NaN,NaN,NaN,NaN,1,234.485223
3,2009,7,23,4,2009-07-23,B6,N-J350,426,PBI,BOS,...,2009-07-23 23:49:00+00:00,204,2009-07-23 18:00:00,17.781553,NaN,NaN,NaN,NaN,1,234.485223
4,2009,7,31,5,2009-07-31,B6,N-J350,307,IAD,LGB,...,2009-08-01 06:45:00+00:00,212,2009-07-31 20:00:00,14.888270,NaN,NaN,NaN,NaN,1,234.485223
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6069427,2010,12,30,4,2010-12-30,UA,NaN,258,DEN,ORD,...,NaT,364,2010-12-31 02:00:00,-0.240415,NaN,NaN,NaN,NaN,1,234.485223
6069428,2010,12,30,4,2010-12-30,UA,NaN,9,DEN,SAN,...,NaT,364,2010-12-31 02:00:00,-0.240415,NaN,NaN,NaN,NaN,1,234.485223
6069429,2010,12,30,4,2010-12-30,F9,NaN,788,DEN,MKE,...,NaT,364,2010-12-31 02:00:00,-0.240415,NaN,NaN,NaN,NaN,1,234.485223
6069430,2010,12,31,5,2010-12-31,UA,NaN,468,SEA,SFO,...,NaT,365,2010-12-31 12:00:00,0.913995,NaN,NaN,NaN,NaN,1,234.485223


In [16]:

dataset = dataset[~dataset['CRSElapsedTime'].isna()]

dataset = dataset[dataset["Origin"].isin(airport_limit_list) & dataset["Dest"].isin(airport_limit_list)]
columns_with_nan_allowed = ['Tail_Number', 'TurnaroundTime','NASDelay','SecurityDelay','LateAircraftDelay','CarrierDelay','WeatherDelay',]
no_cancelled_diverted = dataset[~((dataset['Diverted'] == 1) | (dataset['Cancelled'] == 1))]
for col in dataset.columns:
    unique_values = dataset[col].nunique()
    missing_values = dataset[col].isna().sum()
    missing_values_no_cancelled_diverted = no_cancelled_diverted[col].isna().sum()
    if missing_values > 0:
        print(f"{col}:  Unique:   {unique_values} ---  Missing: {missing_values} --- really Missing: {missing_values_no_cancelled_diverted}")
        if missing_values_no_cancelled_diverted > 0 and col not in columns_with_nan_allowed:
            print(f"Column {col} has {missing_values_no_cancelled_diverted} missing values that are not due to cancelled or diverted flights, which is a problem for the analysis.")


Tail_Number:  Unique:   2813 ---  Missing: 9291 --- really Missing: 0
DepTime:  Unique:   1398 ---  Missing: 45858 --- really Missing: 0
DepDelay:  Unique:   878 ---  Missing: 45858 --- really Missing: 0
DepDelayMinutes:  Unique:   841 ---  Missing: 45858 --- really Missing: 0
TaxiOut:  Unique:   263 ---  Missing: 46948 --- really Missing: 0
WheelsOff:  Unique:   1397 ---  Missing: 46948 --- really Missing: 0
WheelsOn:  Unique:   1440 ---  Missing: 47146 --- really Missing: 0
TaxiIn:  Unique:   159 ---  Missing: 48054 --- really Missing: 0
CRSArrTime:  Unique:   1276 ---  Missing: 7652 --- really Missing: 0
ArrTime:  Unique:   1440 ---  Missing: 54798 --- really Missing: 0
ArrDelay:  Unique:   927 ---  Missing: 54798 --- really Missing: 0
ArrDelayMinutes:  Unique:   828 ---  Missing: 54798 --- really Missing: 0
ActualElapsedTime:  Unique:   621 ---  Missing: 54798 --- really Missing: 0
AirTime:  Unique:   596 ---  Missing: 54798 --- really Missing: 0
CarrierDelay:  Unique:   751 ---  M

In [17]:
target_like = ['DepDelay','DepDelayMinutes','ActualElapsedTime', 'Diverted', 'Cancelled','TaxiOut','TaxiIn','ArrTime','DepTime','ArrDateTime','DepDateTime','AirTime',
               'NASDelay','SecurityDelay','LateAircraftDelay','CarrierDelay','WeatherDelay','ArrDateTime_UTC','WheelsOff','WheelsOn','ArrDelay']
timezone_cols = ['Timezone_Origin','TZ_Origin', 'Timezone_Dest', 'TZ_Dest']
timestamp_cols = ['DepDateTime_UTC', 'CRSArrDateTime_UTC', 'CRSDepTime', 'CRSArrTime','FlightDate']

# drop against overfitting 
overtting_columns = ['FlightId','Tail_Number','Flight_Number_Reporting_Airline']
# drop all canncelled and diverted flights, as they have a lot of missing values in the arrival delay column, and can not be used for training
dataset = dataset[~((dataset['Diverted'] == 1) | (dataset['Cancelled'] == 1))]

# drop all the target like columns from the dataset, as they can not be used as features
dataset = dataset.drop(columns=target_like+timezone_cols+overtting_columns+timestamp_cols)


# creta a feature for the day of the year


In [18]:
dataset.to_feather('all_flights_preprocessed1.feather')